In [ ]:
# BSIT 3A-G1 Group 1
# Adriano, Arianne Denisse
# Cabataña, Jocky King
# Elopre, Ross Anthony
# Pelayo, Danielle Griz
# Zaldivar, Jaen Royce

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   -- ------------------------------------- 0.5/9.5 MB 230.2 kB/s eta 0:00:40
   --- 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import pandas as pd
import numpy as np

df = pd.read_csv('group1_ecommerce_returns_extended_dirty.csv')
df_clean = df.copy()
print("Before:", df_clean.shape, "| nulls:", df_clean.isna().sum().sum())

Before: (10000, 13) | nulls: 1194


In [35]:
# Inconsistent Text
for col in ['Customer Gender', 'Product Category']:
    df_clean[col] = df_clean[col].str.strip().str.title()

df_clean['Customer Gender'] = df_clean['Customer Gender'].replace({'M': 'Male', 'F': 'Female'})
print(df_clean['Customer Gender'].value_counts())
print(df_clean['Product Category'].value_counts())

Customer Gender
Male                 4554
Female               4467
Prefer Not To Say     979
Name: count, dtype: int64
Product Category
Clothing       3493
Electronics    2384
Home           1606
Beauty         1470
Sports         1047
Name: count, dtype: int64


In [36]:
# Date Format Fix
def convert_date(s):
    if '/' in s:
        return pd.to_datetime(s, format='%m/%d/%Y', errors='coerce')
    if s[2] == '-':
        return pd.to_datetime(s, format='%d-%m-%Y', errors='coerce')
    return pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')

df_clean['Date'] = df_clean['Date'].astype(str).str.strip().apply(convert_date)
print('Uncleaned dates:', df_clean['Date'].isna().sum())

Uncleaned dates: 0


In [37]:
# Extreme Values to NaN
df_clean.loc[~df_clean['Customer Age'].between(18, 100), 'Customer Age'] = np.nan
df_clean.loc[df_clean['Total Order Value'] <= 0, 'Total Order Value'] = np.nan
df_clean['Discount Applied'] = df_clean['Discount Applied'].abs()

In [38]:
# Missing Values
df_clean['Rating Missing'] = df_clean['Customer Rating'].isna().astype(int)

df_clean['Customer Rating'] = df_clean['Customer Rating'].fillna(df_clean['Customer Rating'].median()) # median imputation since median is not affected by extreme values
df_clean['Customer Age'] = df_clean['Customer Age'].fillna(df_clean['Customer Age'].median())
df_clean['Total Order Value'] = df_clean.groupby('Product Category')['Total Order Value'].transform(lambda x: x.fillna(x.median()))

In [39]:
# Outlier Check and Verification
for col in ['Customer Age', 'Total Order Value']:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df_clean[col] < q1 - 1.5 * iqr) | (df_clean[col] > q3 + 1.5 * iqr)).sum()
    print(col, "IQR outliers:", n)

print("After:", df_clean.shape, "| nulls:", df_clean.isna().sum().sum())
df_clean.describe().T

Customer Age IQR outliers: 0
Total Order Value IQR outliers: 0
After: (10000, 14) | nulls: 0


,count,mean,min,25%,50%,75%,max,std
Date,10000,2024-05-16 15:54:51.840000,2023-01-01 00:00:00,2023-09-14 00:00:00,2024-05-20 12:00:00,2025-01-15 00:00:00,2025-09-27 00:00:00,NaN
Customer Age,10000.0,43.5162,18.0,31.0,43.0,56.0,69.0,14.753105
Total Order Value,10000.0,12654.593928,500.14,6863.3775,12708.48,18325.9275,24994.84,6861.428204
Discount Applied,10000.0,0.251241,0.0,0.13,0.25,0.38,0.5,0.144418
Customer Rating,10000.0,3.4061,1.0,3.0,4.0,4.0,5.0,1.323086
Rating Missing,10000.0,0.08,0.0,0.0,0.0,0.0,1.0,0.271307


In [ ]:
df_clean.shape[0]